# X-13 ARIMA-SEATS: Ajuste Sazonal Oficial

Este notebook apresenta o metodo **X-13 ARIMA-SEATS** para ajuste sazonal usando a biblioteca `chronobox`.

O X-13 ARIMA-SEATS e o metodo oficial usado por agencias estatisticas (US Census Bureau, IBGE, Eurostat) para dessazonalizacao de series economicas. Combina dois procedimentos:

- **X-11**: filtros de medias moveis iterativos
- **SEATS**: decomposicao baseada em modelo ARIMA

O chronobox fornece um **wrapper** (`X13Wrapper`) que invoca o executavel X-13 externo.

## Setup e Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox import STL, ClassicalDecomposition
from chronobox.decomposition import X13Wrapper
from chronobox.visualization import plot_decomposition

import sys
sys.path.insert(0, '..')
from utils.plot_helpers import plot_decomposition as plot_decomp_helper

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)

## 1. O que e X-13 ARIMA-SEATS?

O X-13 ARIMA-SEATS e desenvolvido pelo US Census Bureau e combina:

### X-11 (Filtros de Medias Moveis)
- Aplica medias moveis simetricas iterativamente
- Henderson trend filter (13 ou 23 termos)
- Medias moveis sazonais (3x3, 3x5, 3x9)
- Deteccao e substituicao automatica de outliers
- Nao requer modelo parametrico

### SEATS (Signal Extraction in ARIMA Time Series)
- Baseado em modelo: ajusta ARIMA sazonal (SARIMA) a serie
- Decompoe via filtro de Wiener-Kolmogorov
- Cada componente e um processo ARIMA
- Minimiza revisoes quando novos dados chegam

### X-11 vs SEATS

| Aspecto | X-11 | SEATS |
|---------|------|-------|
| Abordagem | Filtros ad hoc | Baseado em modelo |
| Flexibilidade | Alta | Media |
| Interpretabilidade | Empirica | Teorica |
| Revisoes | Maiores | Menores |
| Uso tipico | Maioria dos paises | ECB, Banco de Espanha |

## 2. Verificando Disponibilidade do X-13

O X-13 requer um executavel externo. Vamos verificar se esta instalado.

In [ ]:
# Verificar disponibilidade do X-13
x13 = X13Wrapper()
print(f"X-13 disponivel: {x13.is_available}")

if not x13.is_available:
    print("\nATENCAO: X-13 nao esta instalado neste sistema.")
    print("Para instalar: https://www.census.gov/data/software/x13as.html")
    print("\nEste notebook continuara demonstrando o uso conceitual e")
    print("comparando STL como alternativa.")

X13_AVAILABLE = x13.is_available

## 3. Carregando Dados IPCA

Usaremos o dataset sintetico de inflacao mensal brasileira (IPCA) para demonstracao.

In [ ]:
# Carregar dados IPCA
ipca = pd.read_csv('../data/brazil_ipca.csv', parse_dates=['date'])
y_ipca = ipca['ipca'].values
dates_ipca = pd.DatetimeIndex(ipca['date'])

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(dates_ipca, y_ipca, 'k-', linewidth=0.8)
axes[0].set_title('IPCA Mensal (% a.m.) - Dados Sinteticos')
axes[0].set_ylabel('IPCA (% a.m.)')
axes[0].axhline(y_ipca.mean(), color='red', linestyle='--', linewidth=0.5, label=f'Media: {y_ipca.mean():.2f}%')
axes[0].legend()

# Acumulado 12 meses
ipca_12m = pd.Series(y_ipca).rolling(12).apply(lambda x: np.prod(1 + x/100) * 100 - 100)
axes[1].plot(dates_ipca, ipca_12m, 'b-', linewidth=1)
axes[1].set_title('IPCA Acumulado 12 Meses (%)')
axes[1].set_ylabel('IPCA 12m (%)')

fig.tight_layout()
plt.show()

print(f"Observacoes: {len(y_ipca)}")
print(f"Periodo: {dates_ipca[0].strftime('%Y-%m')} a {dates_ipca[-1].strftime('%Y-%m')}")

## 4. Uso do X-13 com chronobox

O `X13Wrapper` do chronobox invoca o executavel X-13 e retorna um `DecompositionResult` compativel com os demais metodos.

In [ ]:
if X13_AVAILABLE:
    # Ajuste sazonal com X-13
    result_x13 = x13.seasonal_adjust(
        endog=y_ipca,
        period=12,
        start_year=2004,
        start_month=1,
        transform='auto'  # auto detecta se precisa log
    )
    
    print(result_x13.summary())
    
    fig = plot_decomposition(result_x13, title='X-13 ARIMA-SEATS - IPCA', dates=dates_ipca)
    plt.show()
else:
    print("X-13 nao disponivel. Usando STL como proxy para demonstracao.")
    stl_proxy = STL(period=12, seasonal=13, robust=True)
    result_x13_proxy = stl_proxy.fit(y_ipca)
    
    fig = plot_decomposition(result_x13_proxy, title='STL (proxy para X-13) - IPCA', dates=dates_ipca)
    plt.show()

### Interpretacao Visual

- **Observed**: IPCA mensal com sazonalidade e nivel variavel
- **Trend**: tendencia inflacionaria capturando ciclos de politica monetaria
- **Seasonal**: padrao sazonal da inflacao brasileira (educacao em fevereiro, alimentacao no verao, etc.)
- **Remainder**: choques e surpresas inflacionarias nao capturadas pela decomposicao

O ajuste sazonal X-13 e fundamental para a analise de politica monetaria - o Banco Central usa series dessazonalizadas para decisoes de juros.

## 5. API do X13Wrapper

```python
x13 = X13Wrapper(x13_path=None)  # None = auto-detect

result = x13.seasonal_adjust(
    endog=y,           # serie temporal (array)
    period=12,         # periodo sazonal (12=mensal, 4=trimestral)
    start_year=2004,   # ano inicial
    start_month=1,     # mes inicial
    transform='auto'   # 'auto', 'log', ou 'none'
)

# Resultado: DecompositionResult
result.observed    # serie original
result.trend       # tendencia-ciclo
result.seasonal    # componente sazonal
result.remainder   # residuo irregular
result.model       # 'x13'
```

O parametro `transform`:
- `'auto'`: X-13 decide se aplica transformacao logaritmica
- `'log'`: forca transformacao log (multiplicativo)
- `'none'`: sem transformacao (aditivo)

## 6. Comparacao: X-13 vs STL no IPCA

Vamos comparar os resultados do X-13 (ou proxy) com STL em diferentes configuracoes.

In [ ]:
# STL com diferentes configuracoes para comparacao
stl_s7 = STL(period=12, seasonal=7)
stl_s13 = STL(period=12, seasonal=13, robust=True)
stl_s25 = STL(period=12, seasonal=25)

res_s7 = stl_s7.fit(y_ipca)
res_s13 = stl_s13.fit(y_ipca)
res_s25 = stl_s25.fit(y_ipca)

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
fig.suptitle('Comparacao de Metodos de Decomposicao - IPCA', fontsize=14)

# Dados originais
axes[0].plot(dates_ipca, y_ipca, 'k-', linewidth=0.8)
axes[0].set_title('Serie Original')
axes[0].set_ylabel('IPCA (%)')

# Tendencias
axes[1].plot(dates_ipca, res_s7.trend, 'b-', label='STL s=7', linewidth=1)
axes[1].plot(dates_ipca, res_s13.trend, 'r-', label='STL s=13 (robust)', linewidth=1)
axes[1].plot(dates_ipca, res_s25.trend, 'g-', label='STL s=25', linewidth=1)
if X13_AVAILABLE:
    axes[1].plot(dates_ipca, result_x13.trend, 'k--', label='X-13', linewidth=1.5)
axes[1].set_title('Tendencia')
axes[1].legend()

# Sazonal
axes[2].plot(dates_ipca, res_s7.seasonal, 'b-', label='STL s=7', linewidth=0.8)
axes[2].plot(dates_ipca, res_s13.seasonal, 'r-', label='STL s=13 (robust)', linewidth=0.8)
axes[2].plot(dates_ipca, res_s25.seasonal, 'g-', label='STL s=25', linewidth=0.8)
if X13_AVAILABLE:
    axes[2].plot(dates_ipca, result_x13.seasonal, 'k--', label='X-13', linewidth=1.5)
axes[2].set_title('Componente Sazonal')
axes[2].legend()

# Residuos
axes[3].plot(dates_ipca, res_s7.remainder, 'b-', label='STL s=7', linewidth=0.8, alpha=0.7)
axes[3].plot(dates_ipca, res_s13.remainder, 'r-', label='STL s=13 (robust)', linewidth=0.8, alpha=0.7)
if X13_AVAILABLE:
    axes[3].plot(dates_ipca, result_x13.remainder, 'k--', label='X-13', linewidth=1.5)
axes[3].axhline(0, color='gray', linestyle='--', linewidth=0.5)
axes[3].set_title('Residuos')
axes[3].legend()

fig.tight_layout()
plt.show()

In [ ]:
# Metricas comparativas
methods = [
    ('STL s=7', res_s7),
    ('STL s=13 robust', res_s13),
    ('STL s=25', res_s25),
]
if X13_AVAILABLE:
    methods.append(('X-13', result_x13))

print(f"{'Metodo':<20} {'Resid Std':>10} {'Resid MAD':>10} {'Seas Range':>12} {'Trend Range':>12}")
print('-' * 68)

for name, res in methods:
    r = res.remainder
    valid = ~np.isnan(r)
    r_std = np.std(r[valid])
    r_mad = np.median(np.abs(r[valid] - np.median(r[valid])))
    s_range = np.nanmax(res.seasonal) - np.nanmin(res.seasonal)
    t_range = np.nanmax(res.trend) - np.nanmin(res.trend)
    print(f"{name:<20} {r_std:>10.4f} {r_mad:>10.4f} {s_range:>12.4f} {t_range:>12.4f}")

### Interpretacao da Comparacao

- **STL s=7**: sazonalidade mais flexivel, tendencia pode capturar variacao excessiva
- **STL s=13 robusto**: bom equilibrio entre flexibilidade e estabilidade
- **STL s=25**: sazonalidade quase fixa, proxima da classica
- **X-13** (se disponivel): usa modelo ARIMA para otimizar a decomposicao; geralmente considerado o padrao-ouro para dados economicos oficiais

Para series economicas oficiais, o X-13 e preferido por sua consistencia e menor revisao quando novos dados sao adicionados.

## 7. Quando Usar Cada Metodo?

| Criterio | Classica | STL | X-13 |
|----------|----------|-----|------|
| **Simplicidade** | Alta | Media | Baixa |
| **Flexibilidade** | Baixa | Alta | Media |
| **Outliers** | Sensivel | Robusto | Detecta e corrige |
| **Dados oficiais** | Nao recomendado | Aceitavel | Padrao-ouro |
| **Dependencia externa** | Nenhuma | Nenhuma | Requer executavel |
| **Multiplas sazonalidades** | Nao | Sim (MSTL) | Limitado |
| **Serie curta** | >= 2 periodos | >= 2 periodos | >= 3 anos (mensal) |

**Recomendacoes praticas:**
- **Analise exploratoria**: comece com decomposicao classica para intuicao
- **Pesquisa/modelagem**: use STL pela flexibilidade
- **Dados oficiais/publicacao**: use X-13 ARIMA-SEATS

## 8. Analise da Sazonalidade do IPCA

Vamos examinar o padrao sazonal da inflacao brasileira em mais detalhe.

In [ ]:
# Padrao sazonal medio usando STL robusto
seasonal_ipca = res_s13.seasonal
month_names = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
               'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

# Media sazonal por mes
seasonal_by_month = np.array([seasonal_ipca[i::12].mean() for i in range(12)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fatores sazonais medios
colors = ['steelblue' if v >= 0 else 'salmon' for v in seasonal_by_month]
axes[0].bar(month_names, seasonal_by_month, color=colors, edgecolor='gray')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Fator Sazonal Medio - IPCA')
axes[0].set_ylabel('Efeito Sazonal (p.p.)')

# Evolucao sazonal ao longo do tempo
n_years = len(y_ipca) // 12
seasonal_matrix = seasonal_ipca[:n_years*12].reshape(n_years, 12)
years = [dates_ipca[i*12].year for i in range(n_years)]

im = axes[1].imshow(seasonal_matrix, aspect='auto', cmap='RdBu_r',
                     vmin=-seasonal_matrix.max(), vmax=seasonal_matrix.max())
axes[1].set_yticks(range(n_years))
axes[1].set_yticklabels(years)
axes[1].set_xticks(range(12))
axes[1].set_xticklabels(month_names, rotation=45)
axes[1].set_title('Evolucao dos Fatores Sazonais')
plt.colorbar(im, ax=axes[1], label='Efeito Sazonal')

fig.tight_layout()
plt.show()

### Interpretacao do Padrao Sazonal IPCA

O padrao sazonal sintetico do IPCA reflete caracteristicas tipicas:
- **Janeiro/Fevereiro**: pressao inflacionaria (reajustes de inicio de ano, mensalidades escolares)
- **Maio-Julho**: tendencia de menor inflacao (safra agricola)
- **Novembro/Dezembro**: pressao de final de ano (demanda aquecida)

O heatmap mostra como esse padrao pode variar ano a ano - o STL captura essa variacao.

## 9. Serie Dessazonalizada

A serie dessazonalizada (Trend + Remainder) e a principal saida do ajuste sazonal X-13.

In [ ]:
# Serie dessazonalizada: Y - S = T + R
y_sa = y_ipca - res_s13.seasonal

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(dates_ipca, y_ipca, 'k-', linewidth=0.5, alpha=0.5, label='Original')
ax.plot(dates_ipca, y_sa, 'b-', linewidth=1, label='Dessazonalizada')
ax.plot(dates_ipca, res_s13.trend, 'r-', linewidth=1.5, label='Tendencia')
ax.set_title('IPCA: Original vs Dessazonalizada vs Tendencia')
ax.set_ylabel('IPCA (% a.m.)')
ax.legend()
plt.show()

print("A serie dessazonalizada remove o padrao sazonal, facilitando")
print("a analise de movimentos de tendencia e choques.")
print(f"\nVolatilidade (std):")
print(f"  Original:         {y_ipca.std():.4f}")
print(f"  Dessazonalizada:  {y_sa.std():.4f}")
print(f"  Reducao:          {(1 - y_sa.std()/y_ipca.std())*100:.1f}%")

## Exercicio

### Exercicio 1: Comparacao X-13 vs STL no IPCA

1. Aplique STL ao `brazil_ipca.csv` com 3 configuracoes diferentes
2. Se X-13 estiver disponivel, compare com o ajuste X-13
3. Plote as series dessazonalizadas de cada metodo
4. Calcule a correlacao entre as series dessazonalizadas
5. Qual metodo produz a serie dessazonalizada mais suave?

### Exercicio 2: Escolha do Metodo

1. Carregue o `airline.csv` e aplique os 3 metodos (classica multiplicativa, STL, X-13)
2. Para cada metodo, calcule e compare:
   - Desvio-padrao dos residuos
   - Amplitude sazonal (max - min do componente sazonal)
3. Qual metodo voce recomendaria para esta serie? Justifique.

In [ ]:
# Exercicio 1 - Espaco para resolucao
# Dica: y_sa_stl = y_ipca - result_stl.seasonal
# Para correlacao: np.corrcoef(sa1, sa2)[0, 1]

# Seu codigo aqui:


In [ ]:
# Exercicio 2 - Espaco para resolucao
# Dica: Para o airline, use ClassicalDecomposition(12, 'multiplicative')
# STL opera no modelo aditivo - para series multiplicativas, aplique log

# Seu codigo aqui:
